# 03 — Model Comparison

Compare the statistical baseline (XGBoost), fine-tuned CodeBERT, and LLM-as-judge
detectors on the held-out test set. Overlay ROC curves, compare F1/AUC, and
analyse per-language performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config.settings import MODELS_DIR, SPLITS_DIR
from evaluation.metrics import (
    compute_all_metrics,
    compare_models,
    per_language_metrics,
    per_problem_metrics,
    plot_confusion_matrix,
    plot_precision_recall_curve,
    print_report,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
test_df = pd.read_parquet(SPLITS_DIR / 'test.parquet')
y_true = test_df['label'].values
print(f'Test set: {len(test_df)} samples, {y_true.sum()} AI, {(1-y_true).sum()} human')

In [ ]:
# Load statistical baseline predictions
from src.models.statistical_baseline import build_feature_matrix, get_feature_columns

model_path = MODELS_DIR / 'xgb_baseline.pkl'
xgb_probs = None
if model_path.exists():
    with open(model_path, 'rb') as f:
        xgb_model = pickle.load(f)
    meta = json.loads((MODELS_DIR / 'xgb_baseline_meta.json').read_text())
    feature_cols = meta['feature_columns']
    test_feat = build_feature_matrix(test_df)
    X_test = test_feat[feature_cols].fillna(0).values
    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    print('XGBoost baseline loaded')
    print_report(y_true, xgb_probs)
else:
    print('XGBoost model not found')

In [ ]:
# Load CodeBERT predictions
import torch
from transformers import AutoTokenizer
from src.models.codebert_classifier import CodeBERTClassifier, CodeDataset
from config.settings import CODEBERT_MODEL_NAME, CODEBERT_MAX_LENGTH
from torch.utils.data import DataLoader

codebert_probs = None
codebert_path = MODELS_DIR / 'codebert_final'
if codebert_path.exists():
    tokenizer = AutoTokenizer.from_pretrained(str(codebert_path))
    model = CodeBERTClassifier(CODEBERT_MODEL_NAME)
    state = torch.load(codebert_path / 'pytorch_model.bin', map_location='cpu')
    model.load_state_dict(state)
    model.eval()

    dataset = CodeDataset(test_df, tokenizer, CODEBERT_MAX_LENGTH)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs = []
    with torch.no_grad():
        for batch in loader:
            out = model(batch['input_ids'], batch['attention_mask'])
            probs = torch.sigmoid(out['logits']).cpu().numpy()
            all_probs.extend(probs)
    codebert_probs = np.array(all_probs)
    print('CodeBERT loaded')
    print_report(y_true, codebert_probs)
else:
    print('CodeBERT model not found')

In [ ]:
# Compare models with overlaid ROC curves
model_probs = {}
if xgb_probs is not None:
    model_probs['XGBoost Baseline'] = xgb_probs
if codebert_probs is not None:
    model_probs['CodeBERT'] = codebert_probs

if model_probs:
    fig = compare_models(y_true, model_probs)
    plt.show()
else:
    print('No models to compare')

In [ ]:
# Per-language comparison
for name, probs in model_probs.items():
    test_df[f'prob_{name}'] = probs

for name, probs in model_probs.items():
    print(f'\n=== {name} — Per-Language Metrics ===')
    lang_metrics = per_language_metrics(test_df, f'prob_{name}')
    for lang, m in lang_metrics.items():
        print(f"  {lang}: F1={m['f1']:.3f}, AUC={m.get('auc_roc', 0):.3f}, n={m['n_samples']}")

In [ ]:
# Per-problem difficulty analysis
if model_probs:
    best_model_name = max(model_probs.keys(),
                         key=lambda k: compute_all_metrics(y_true, model_probs[k])['f1'])
    test_df['prob'] = model_probs[best_model_name]
    problem_df = per_problem_metrics(test_df)
    print(f'\nHardest problems to distinguish ({best_model_name}):')
    print(problem_df.head(10)[['problem_id', 'n_samples', 'f1', 'precision', 'recall']])